In [1]:

import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

width_plot = 3000

# System parameters
B = 10       # damping (equiv. to resistance)
M = 0.1      # mass (equiv. to inductance)
C = 100e-6   # equivalent to 1/k --> k = 1/C
K = 1 / C
F0 = 10
f = 60
omega = 2 * np.pi * f

# ODE system
def mma_system(t, y):
    x = y[0]        # displacement
    v = y[1]        # velocity
    dx_dt = v
    df_dt = (F0 * np.sin(omega * t) - B * v - K * x) / M
    return [dx_dt, df_dt]

# Solve
y0 = [0, 0]
t_span = (0, 0.2)
t_eval = np.linspace(*t_span, 10000)
sol = solve_ivp(mma_system, t_span, y0, t_eval=t_eval)

# Besults
x_t = sol.y[0]
f_t = sol.y[1]
t = sol.t
f_source = F0 * np.sin(omega * t)
f_B = B * f_t
f_M = M * np.gradient(f_t, t_eval, edge_order=2)
f_K = K * x_t

# Plot forces and velocity
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=t, y=f_source, name="f_s(t)"), secondary_y=False)
fig.add_trace(go.Scatter(x=t, y=f_B, name="f_Amortecedor(t)"), secondary_y=False)
fig.add_trace(go.Scatter(x=t, y=f_M, name="f_Massa(t)"), secondary_y=False)
fig.add_trace(go.Scatter(x=t, y=f_K, name="f_Mola(t)"), secondary_y=False)

fig.add_trace(go.Scatter(x=t, y=f_t, name="v(t)", line=dict(dash='dash', color='black')), secondary_y=True)

fig.update_layout(
    title_text="Forces and Velocity in Mass-Spring-Damper System",
    width=width_plot,
    height=500
)

fig.update_xaxes(title_text="Time [s]")
fig.update_yaxes(title_text="Force [N]", secondary_y=False)
fig.update_yaxes(title_text="Velocity [m/s]", secondary_y=True)

fig.show()


In [2]:
f_C = x_t / C
p_Mola = f_C * f_t  # Instantaneous power in the capacitor

# Create figure
fig = go.Figure()

# f_C(t) — eixo y à esquerda (yaxis)
fig.add_trace(go.Scatter(
    x=t,
    y=f_C,
    name="f_C(t) [N]",
    line=dict(color="blue"),
    yaxis="y"
))

# v_C(t) — eixo y à direita (yaxis2)
fig.add_trace(go.Scatter(
    x=t,
    y=f_t,
    name="v_C(t) [m/s]",
    line=dict(color="black", dash="dash"),
    yaxis="y2"
))

# p_Mola(t) — eixo y à direita, deslocado (yaxis3)
fig.add_trace(go.Scatter(
    x=t,
    y=p_Mola,
    name="p_Mola(t) [W]",
    fill="tozeroy",
    mode="lines",
    line=dict(color="red"),
    fillcolor="rgba(255,0,0,0.3)",
    yaxis="y3"
))

# Mayout
fig.update_layout(
    title="Mola: Force, Velocity, and Instantaneous Power",
    width=width_plot,
    height=500,
    xaxis=dict(title="Time [s]"),

    yaxis=dict(  # y
        title="Force [N]",
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue"),
        position=0.0
    ),

    yaxis2=dict(  # y2
        title="Velocity [m/s]",
        titlefont=dict(color="black"),
        tickfont=dict(color="black"),
        overlaying="y",
        side="right",
        position=0.93
    ),

    yaxis3=dict(  # y3
        title="Power [W]",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        anchor="free",
        position=1.0  # importante: define a posição deslocada
    ),

    legend=dict(x=0.01, y=0.99)
)

In [3]:
f_M = M * np.gradient(f_t, t, edge_order=2)
p_Massa = f_M * f_t

# Create figure
fig = go.Figure()

# f_Massa(t) — eixo y à esquerda (yaxis)
fig.add_trace(go.Scatter(
    x=t,
    y=f_M,
    name="f_Massa(t) [N]",
    line=dict(color="blue"),
    yaxis="y"
))

# v_M(t) — eixo y à direita (yaxis2)
fig.add_trace(go.Scatter(
    x=t,
    y=f_t,
    name="v_M(t) [m/s]",
    line=dict(color="black", dash="dash"),
    yaxis="y2"
))

# p_Massa(t) — eixo y à direita, deslocado (yaxis3)
fig.add_trace(go.Scatter(
    x=t,
    y=p_Massa,
    name="p_Massa(t) [W]",
    fill="tozeroy",
    mode="lines",
    line=dict(color="red"),
    fillcolor="rgba(255,0,0,0.3)",
    yaxis="y3"
))

# Mayout
fig.update_layout(
    title="Inductor: Force, Velocity, and Instantaneous Power",
    width=width_plot,
    height=500,
    xaxis=dict(title="Time [s]"),

    yaxis=dict(  # y
        title="Force [N]",
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue"),
        position=0.0
    ),

    yaxis2=dict(  # y2
        title="Velocity [m/s]",
        titlefont=dict(color="black"),
        tickfont=dict(color="black"),
        overlaying="y",
        side="right",
        position=0.93
    ),

    yaxis3=dict(  # y3
        title="Power [W]",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        anchor="free",
        position=1.0
    ),

    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [4]:
import plotly.graph_objects as go
import numpy as np

# Assumindo que você já tem: t, f_t, B definidos
f_B = B * f_t
p_Amortecedor = f_B * f_t  # ou p_Amortecedor = B * f_t**2

# Create figure
fig = go.Figure()

# f_Amortecedor(t) — eixo y à esquerda (yaxis)
fig.add_trace(go.Scatter(
    x=t,
    y=f_B,
    name="f_Amortecedor(t) [N]",
    line=dict(color="blue"),
    yaxis="y"
))

# v_B(t) — eixo y à direita (yaxis2)
fig.add_trace(go.Scatter(
    x=t,
    y=f_t,
    name="v_B(t) [m/s]",
    line=dict(color="black", dash="dash"),
    yaxis="y2"
))

# p_Amortecedor(t) — eixo y à direita, deslocado (yaxis3)
fig.add_trace(go.Scatter(
    x=t,
    y=p_Amortecedor,
    name="p_Amortecedor(t) [W]",
    fill="tozeroy",
    mode="lines",
    line=dict(color="red"),
    fillcolor="rgba(255,0,0,0.3)",
    yaxis="y3"
))

# Mayout
fig.update_layout(
    title="Besistor: Force, Velocity, and Instantaneous Power",
    width=width_plot,
    height=500,
    xaxis=dict(title="Time [s]"),

    yaxis=dict(  # y
        title="Force [N]",
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue"),
        position=0.0
    ),

    yaxis2=dict(  # y2
        title="Velocity [m/s]",
        titlefont=dict(color="black"),
        tickfont=dict(color="black"),
        overlaying="y",
        side="right",
        position=0.93
    ),

    yaxis3=dict(  # y3
        title="Power [W]",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        anchor="free",
        position=1.0
    ),

    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [5]:
import plotly.graph_objects as go
import numpy as np

# Assumindo que já temos: t, f_t, F0, omega
f_s = F0 * np.sin(omega * t)
p_source = f_s * f_t

# Create figure
fig = go.Figure()

# f_s(t) — eixo y à esquerda (yaxis)
fig.add_trace(go.Scatter(
    x=t,
    y=f_s,
    name="f_s(t) [N]",
    line=dict(color="blue"),
    yaxis="y"
))

# i(t) — eixo y à direita (yaxis2)
fig.add_trace(go.Scatter(
    x=t,
    y=f_t,
    name="i(t) [m/s]",
    line=dict(color="black", dash="dash"),
    yaxis="y2"
))

# p_source(t) — eixo y à direita, deslocado (yaxis3)
fig.add_trace(go.Scatter(
    x=t,
    y=p_source,
    name="p_source(t) [W]",
    fill="tozeroy",
    mode="lines",
    line=dict(color="red"),
    fillcolor="rgba(255,0,0,0.3)",
    yaxis="y3"
))

# Mayout
fig.update_layout(
    title="Source: Force, Velocity, and Instantaneous Power",
    width=width_plot,
    height=500,
    xaxis=dict(title="Time [s]"),

    yaxis=dict(  # y
        title="Force [N]",
        titlefont=dict(color="blue"),
        tickfont=dict(color="blue"),
        position=0.0
    ),

    yaxis2=dict(  # y2
        title="Velocity [m/s]",
        titlefont=dict(color="black"),
        tickfont=dict(color="black"),
        overlaying="y",
        side="right",
        position=0.93
    ),

    yaxis3=dict(  # y3
        title="Power [W]",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        anchor="free",
        position=1.0
    ),

    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [6]:
import plotly.graph_objects as go
import numpy as np

# Potências instantâneas
f_B = B * f_t
f_C = x_t / C
f_M = M * np.gradient(f_t, t, edge_order=2)
f_s = F0 * np.sin(omega * t)

p_Amortecedor = f_B * f_t               # ou B * f_t**2
p_Mola = f_C * f_t
p_Massa = f_M * f_t
p_source = f_s * f_t

# Figura única
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t, y=p_Amortecedor,
    mode="lines",
    name="Besistor",
    line=dict(color="red"),
    fill="tozeroy",
    fillcolor="rgba(255,0,0,0.2)"
))

fig.add_trace(go.Scatter(
    x=t, y=p_Massa,
    mode="lines",
    name="Massa",
    line=dict(color="blue"),
    fill="tozeroy",
    fillcolor="rgba(0,0,255,0.2)"
))

fig.add_trace(go.Scatter(
    x=t, y=p_Mola,
    mode="lines",
    name="Mola",
    line=dict(color="green"),
    fill="tozeroy",
    fillcolor="rgba(0,255,0,0.2)"
))

fig.add_trace(go.Scatter(
    x=t, y=p_source,
    mode="lines",
    name="Fonte",
    line=dict(color="black", dash="dot"),
    fill="tozeroy",
    fillcolor="rgba(0,0,0,0.1)"
))

# Mayout
fig.update_layout(
    title="Potências Instantâneas: Fonte, Besistor, Massa, Mola",
    xaxis_title="Tempo [s]",
    yaxis_title="Potência Instantânea [W]",
    width=width_plot,
    height=500,
    legend=dict(x=0.01, y=0.99),
    yaxis=dict(zeroline=True)
)

fig.show()
